# Lesson 4: Auto-merging Retrieval (自动合并检索)

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import importlib
import utils
importlib.reload(utils)

import os
import openai

In [3]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_files=["./eBook-How-to-Build-a-Career-in-AI.pdf"]
).load_data()

In [4]:
print(type(documents), "\n")
print(len(documents), "\n")
print(type(documents[0]))
print(documents[0])

<class 'list'> 

41 

<class 'llama_index.core.schema.Document'>
Doc ID: 1a41173f-abd6-4b28-8009-5c3f34a3a029
Text: PAGE 1 Founder, DeepLearning.AI Collected Insights from Andrew
Ng How to  Build Your Career in AI A Simple Guide


## Auto-merging retrieval setup (自动合并检索设置)

In [5]:
from llama_index.core import Document

document = Document(text="\n\n".join([doc.text for doc in documents]))

In [6]:
from llama_index.core.node_parser import HierarchicalNodeParser

# create the hierarchical node parser w/ default settings
# --- 1. 定义层级节点解析器 ---

# 导入层级节点解析器类。
node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[2048, 512, 128]
)

##### 📝 LlamaIndex 笔记：HierarchicalNodeParser (层级节点解析器)

这段代码是构建“自动合并检索”系统的第一步，负责建立文档的层级树状结构。

```python
from llama_index.core.node_parser import HierarchicalNodeParser

# --- 1. 定义层级节点解析器 ---

# 导入层级节点解析器类。
# 该解析器会将文档递归地切分成具有“父子关系”的不同粒度节点。
node_parser = HierarchicalNodeParser.from_defaults(
    # chunk_sizes: 定义层级的粒度。
    # 这里的设置意味着：
    # 第一层 (Root): 每个块 2048 字符 (大背景)
    # 第二层 (Mid): 每个块 512 字符  (中等上下文)
    # 第三层 (Leaf): 每个块 128 字符 (精确检索点)
    chunk_sizes=[2048, 512, 128]
)

In [7]:
# --- 2. 解析文档并建立层级关系 ---

# 对单一 Document 对象执行解析。
# 解析器将根据 chunk_sizes 创建不同大小的节点，并自动建立父子关系。
nodes = node_parser.get_nodes_from_documents([document])

In [8]:
# --- 3. 提取和探索节点关系 ---
from llama_index.core.node_parser import get_leaf_nodes

# 从所有节点中，提取出最细粒度的节点（即最小 chunk size 128 产生的节点）。
# 在 RAG 检索时，通常只对这些“叶节点”进行向量化和检索。
leaf_nodes = get_leaf_nodes(nodes)
print(leaf_nodes[30].text)

Of course, I also encourage learning driven by curiosity. If something interests you, go ahead 
and learn it regardless of how useful it might turn out to be!  Maybe this will lead to a creative 
spark or technical breakthrough.
How much math do you need to know to be a machine learning engineer?


In [9]:
# 创建一个字典，用于通过节点的 node_id 快速查找节点对象。
nodes_by_id = {node.node_id: node for node in nodes}

# 获取第 30 个叶节点的父节点 ID。
# 使用父节点 ID 从字典中获取父节点对象（中等粒度 512 的文本块）。
parent_node = nodes_by_id[leaf_nodes[30].parent_node.node_id]
# 打印父节点的文本内容。
# 这个文本应该比叶节点的文本长得多，演示了层级结构中父节点提供的更广阔的上下文。
print(parent_node.text)

On some days, maybe you’ll end up studying for an 
hour or longer.

PAGE 12
Should You 
Learn Math to 
Get a Job in AI? 
CHAPTER 3
LEARNING

PAGE 13
Should you Learn Math to Get a Job in AI? CHAPTER 3
Is math a foundational skill for AI? It’s always nice to know more math! But there’s so much to 
learn that, realistically, it’s necessary to prioritize. Here’s how you might go about strengthening 
your math background.
To figure out what’s important to know, I find it useful to ask what you need to know to make 
the decisions required for the work you want to do. At DeepLearning.AI, we frequently ask, 
“What does someone need to know to accomplish their goals?” The goal might be building a 
machine learning model, architecting a system, or passing a job interview.
Understanding the math behind algorithms you use is often helpful, since it enables you to 
debug them. But the depth of knowledge that’s useful changes over time. As machine learning 
techniques mature and become more reliabl

### Building the index (建立索引)

In [10]:
from llama_index.llms.openai_like import OpenAILike

# OpenAILike: LlamaIndex 中用于连接与 OpenAI API 兼容的服务的类
# 这里用于连接阿里云的通义千问 (DashScope) 服务
#    配置了 API Key、基础 URL (api_base) 和模型 (qwen-max)。
#    设置了较低的 temperature=0.1，以获得更确定性的回答。
#    设置了较大的 context_window=128000。
llm = OpenAILike(
    api_key=utils.get_dashscope_api_key(),
    api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-max",
    temperature=0.1,
    context_window=128000,
    is_chat_model=True,
    is_function_calling_model=False,
)

In [15]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding  # 用于加载本地或远程的 HuggingFace 嵌入模型
model_real_path = os.path.expanduser(
    "~/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"
)

# Settings: LlamaIndex 的全局配置对象
# 将全局的语言模型设置为配置好的 OpenAILike 实例
Settings.llm = llm
# 将全局的嵌入模型设置为本地加载的 BGE-Small 模型，用于将文本内容转化为向量
Settings.embed_model = HuggingFaceEmbedding(
    model_name=model_real_path,
    # model_name="BAAI/bge-small-en-v1.5",
    # 如果您的机器有 GPU，建议设置 device="cuda"
    device="cpu", 
    # 连不了外网记得这个标志要设置为True，不然虽然本地有了还会掉huggingface获取包信息检验
    local_files_only=True,
)
# 设置节点解析器,节点解析器负责将原始文档（Document）分割成更小的、可管理的块，这些块被称为节点（Nodes）。
Settings.node_parser = node_parser


2026-04-25 15:57:58,681 - INFO - Load pretrained SentenceTransformer: /Users/a1-6/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a


In [16]:
from llama_index.core import VectorStoreIndex, StorageContext
# --- 1. 自动合并索引的首次创建与持久化 ---

# 创建一个默认的存储上下文（Storage Context）。
# 存储上下文是 LlamaIndex 中用于管理数据存储和持久化的核心组件。
storage_context = StorageContext.from_defaults()
# 将所有节点（nodes，包括叶节点和父节点，来自 HierarchicalNodeParser）添加到文档存储（DocStore）中。
# 这是自动合并索引的关键：DocStore 必须包含所有层级的节点，以便进行合并。
storage_context.docstore.add_documents(nodes)

# 创建 VectorStoreIndex。
# 注意：索引的向量存储只基于最细粒度的“叶节点”（leaf_nodes），因为检索应该在最精确的粒度上进行。
automerging_index = VectorStoreIndex(
    leaf_nodes, storage_context=storage_context
)

# 将索引的存储上下文（包括节点、向量和元数据）持久化到本地目录。
automerging_index.storage_context.persist(persist_dir="./merging_index")

In [17]:
# This block of code is optional to check
# if an index file exist, then it will load it
# if not, it will rebuild it

# --- 2. 索引的创建或加载逻辑（避免重复创建） ---
import os
from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.core import load_index_from_storage

# 检查持久化目录是否存在。
if not os.path.exists("./merging_index"):
    # 如果不存在，则重建索引（与上面的代码逻辑相同）。
    storage_context = StorageContext.from_defaults()
    storage_context.docstore.add_documents(nodes) # 确保所有节点在 DocStore 中

    automerging_index = VectorStoreIndex(
        leaf_nodes,
        storage_context=storage_context
        # 假设 llm 和 embed_model 已通过 Settings 设置
    )

    automerging_index.storage_context.persist(persist_dir="./merging_index")
else:
    # 如果存在，则加载已持久化的索引。
    automerging_index = load_index_from_storage(
        StorageContext.from_defaults(persist_dir="./merging_index")
        # 假设 llm 和 embed_model 已通过 Settings 设置
    )


2026-04-25 16:00:54,302 - INFO - Loading all indices.


### Defining the retriever and running the query engine (定义检索器并运行查询引擎)

In [18]:
from llama_index.core.indices.postprocessor import SentenceTransformerRerank
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

# --- 3. 配置和构建自动合并查询引擎 ---

# 1. 创建基础检索器：这是一个标准的 Vector Store Retriever，但作用于叶节点。
automerging_retriever = automerging_index.as_retriever(
    similarity_top_k=12
)

# 2. 包装成 AutoMergingRetriever（自动合并检索器）。
# 这是 RAG 自动合并策略的核心组件。
retriever = AutoMergingRetriever(
    automerging_retriever, 
    automerging_index.storage_context, 
    verbose=True
)

# 3. 创建重排序器（Reranker）。
reranker_base_path = os.path.expanduser("~/Desktop/AIAgent/models/models--BAAI--bge-reranker-base/snapshots/2cfc18c9415c912f9d8155881c133215df768a70")
rerank = SentenceTransformerRerank(top_n=6, model=reranker_base_path)

# 4. 创建查询引擎：使用 RetrieverQueryEngine.from_args 组装检索器和后处理器。
auto_merging_engine = RetrieverQueryEngine.from_args(
    # 🚨 注意：这里传入了基础检索器 automerging_retriever，
    # 但如果目的是使用自动合并功能，通常应该传入上面定义的 AutoMergingRetriever 实例。
    # 假设此处的 intention 是使用 AutoMergingRetriever 及其合并逻辑，但代码传入了基础检索器。
    # **为避免错误，下面的注释将假设代码作者的目的是使用 automerging_retriever 作为 RAG 的 Retriever，
    # 但实际的 Automerging 机制可能隐含在 automerging_index.as_retriever 的默认行为中，或代码有遗漏。
    # 如果要使用 AutoMergingRetriever 实例，这里应是: retriever**
    automerging_retriever, node_postprocessors=[rerank]
)

In [21]:
## --- 4. 执行查询与结果展示 ---

# 执行查询。
# 查询流程：
# 1. automerging_retriever 检索叶节点。
# 2. (如果使用 AutoMergingRetriever) 检索到的叶节点被合并成它们的父节点（较大的上下文）。
# 3. [rerank] 对节点进行重排序和 top_n 筛选。
# 4. 将最终节点作为上下文传递给 LLM 生成答案
auto_merging_response = auto_merging_engine.query(
    "What is the importance of networking in AI?"
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-25 16:04:45,727 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


In [22]:
from llama_index.core.response.notebook_utils import display_response

# 在 Notebook 环境中以美观的方式显示 LLM 的响应。
display_response(auto_merging_response)

**`Final Response:`** Networking in AI is important because it can help you connect with others in the field, which can be beneficial for both personal and professional growth. It allows you to find support, gain advice, and potentially open doors to new opportunities such as jobs or projects. Building a strong professional network can provide you with the resources and assistance you need to advance in your career. Additionally, being part of a community can lead to making friends and meeting like-minded individuals who share your interests and can offer valuable insights and collaboration.

## Putting it all Together (联合起来)

In [31]:
import os

from llama_index.core import (
    ServiceContext,
    StorageContext,
    VectorStoreIndex,
    load_index_from_storage,
)
from llama_index.core.node_parser import HierarchicalNodeParser
from llama_index.core.node_parser import get_leaf_nodes
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core.indices.postprocessor import SentenceTransformerRerank
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.settings import Settings

def build_automerging_index(
    documents,
    llm,
    embed_model,
    save_dir="merging_index",
    chunk_sizes=None,
):
    chunk_sizes = chunk_sizes or [2048, 512, 128]
    node_parser = HierarchicalNodeParser.from_defaults(chunk_sizes=chunk_sizes)
    nodes = node_parser.get_nodes_from_documents(documents)
    leaf_nodes = get_leaf_nodes(nodes)
    Settings.llm = llm
    Settings.embed_model = HuggingFaceEmbedding(
        # model_name=LOCAL_BGE_PATH,
        model_name=embed_model,
        # 如果您的机器有 GPU，建议设置 device="cuda"
        device="mps", 
        # 连不了外网记得这个标志要设置为True，不然虽然本地有了还会掉huggingface获取包信息检验
        local_files_only=True,
    )
    Settings.node_parser = node_parser
    
    storage_context = StorageContext.from_defaults()
    storage_context.docstore.add_documents(nodes)

    if not os.path.exists(save_dir):
        automerging_index = VectorStoreIndex(
            leaf_nodes, storage_context=storage_context
        )
        automerging_index.storage_context.persist(persist_dir=save_dir)
    else:
        automerging_index = load_index_from_storage(
            StorageContext.from_defaults(persist_dir=save_dir)
        )
    return automerging_index


def get_automerging_query_engine(
    automerging_index,
    similarity_top_k=12,
    rerank_top_n=6,
):
    base_retriever = automerging_index.as_retriever(similarity_top_k=similarity_top_k)
    retriever = AutoMergingRetriever(
        base_retriever, automerging_index.storage_context, verbose=True
    )
    reranker_base_path = os.path.expanduser("~/Desktop/AIAgent/models/models--BAAI--bge-reranker-base/snapshots/2cfc18c9415c912f9d8155881c133215df768a70")
    rerank = SentenceTransformerRerank(
        top_n=rerank_top_n, model=reranker_base_path
    )
    auto_merging_engine = RetrieverQueryEngine.from_args(
        retriever, node_postprocessors=[rerank]
    )
    return auto_merging_engine

In [32]:
from llama_index.llms.openai_like import OpenAILike

# OpenAILike: LlamaIndex 中用于连接与 OpenAI API 兼容的服务的类
# 这里用于连接阿里云的通义千问 (DashScope) 服务
#    配置了 API Key、基础 URL (api_base) 和模型 (qwen-max)。
#    设置了较低的 temperature=0.1，以获得更确定性的回答。
#    设置了较大的 context_window=128000。
llm = OpenAILike(
    api_key=utils.get_dashscope_api_key(),
    api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-max",
    temperature=0.1,
    context_window=128000,
    is_chat_model=True,
    is_function_calling_model=False,
)
model_real_path = os.path.expanduser(
    "~/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"
)
index = build_automerging_index(
    [document],
    llm=llm,
    embed_model=model_real_path,
    save_dir="./merging_index",
)


2026-04-25 16:12:51,666 - INFO - Load pretrained SentenceTransformer: /Users/a1-6/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a
2026-04-25 16:12:51,963 - INFO - Loading all indices.


In [33]:
query_engine = get_automerging_query_engine(index, similarity_top_k=6)

## TruLens Evaluation

In [34]:
from trulens.core import Tru 

Tru().reset_database()

2026-04-25 16:13:57,049 - INFO - Context impl SQLiteImpl.
2026-04-25 16:13:57,049 - INFO - Will assume non-transactional DDL.
2026-04-25 16:13:57,079 - INFO - ✅ OpenTelemetry exporter set: NoneType
2026-04-25 16:13:57,110 - INFO - ✅ Added new TrulensOtelSpanProcessor
2026-04-25 16:13:57,119 - INFO - Instrumenting AsyncBatches.create for cost tracking
2026-04-25 16:13:57,121 - INFO - Instrumenting AsyncCompletions.create for cost tracking
2026-04-25 16:13:57,122 - INFO - Instrumenting AsyncContainers.create for cost tracking
2026-04-25 16:13:57,123 - INFO - Instrumenting AsyncEmbeddings.create for cost tracking
2026-04-25 16:13:57,125 - INFO - Instrumenting AsyncEvals.create for cost tracking
2026-04-25 16:13:57,125 - INFO - Instrumenting AsyncFiles.create for cost tracking
2026-04-25 16:13:57,127 - INFO - Instrumenting AsyncModerations.create for cost tracking
2026-04-25 16:13:57,128 - INFO - Instrumenting AsyncUploads.create for cost tracking
2026-04-25 16:13:57,128 - INFO - Instrumen

🦑 Initialized with db url sqlite:///default.sqlite .
🛑 Secret keys may be written to the database. See the `database_redact_keys` option of `TruSession` to prevent this.
✅ experimental Feature.OTEL_TRACING enabled.
🔒 experimental Feature.OTEL_TRACING is enabled and cannot be changed.


Updating app_name and app_version in apps table: 0it [00:00, ?it/s]
Updating app_id in records table: 0it [00:00, ?it/s]
Updating app_json in apps table: 0it [00:00, ?it/s]


### Two layers (两层)

In [35]:
model_real_path = os.path.expanduser("~/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"
)
auto_merging_index_0 = build_automerging_index(
    documents,
    llm=llm,
    embed_model=model_real_path,
    save_dir="merging_index_0",
    chunk_sizes=[2048,512],
)

In [36]:
auto_merging_engine_0 = get_automerging_query_engine(
    auto_merging_index_0,
    similarity_top_k=12,
    rerank_top_n=6,
)

In [37]:
from utils import get_prebuilt_trulens_recorder

tru_recorder = get_prebuilt_trulens_recorder(
    auto_merging_engine_0,
    app_id ='app_0'
)

instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.embeddings.multi_modal_base.MultiModalEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.base.embeddings.base.BaseEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.schema.TransformComponent'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.schema.BaseComponent'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'pydantic.main.BaseModel'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base

In [47]:
eval_questions = []
with open('generated_questions.text', 'r') as file:
    for line in file:
        # Remove newline character and convert to integer
        item = line.strip()
        eval_questions.append(item)

In [46]:
def run_evals(eval_questions, tru_recorder, query_engine):
    for question in eval_questions:
        with tru_recorder as recording:
            response = query_engine.query(question)

In [48]:
run_evals(eval_questions, tru_recorder, auto_merging_engine_0)

> Merging 1 nodes into parent node.
> Parent node id: 1361fd00-b579-40b1-aad4-b60d49e64458.
> Parent node text: PAGE 26
If you’re considering a role switch, a startup can be an easier place to do it than a big...

> Merging 1 nodes into parent node.
> Parent node id: 61bddc0c-e133-445a-b7bb-35cf6964e2fd.
> Parent node text: PAGE 25
Finding a job has a few predictable steps that include selecting the companies to which y...

> Merging 1 nodes into parent node.
> Parent node id: fc1dda3c-5696-445b-922e-6eda94da14b6.
> Parent node text: PAGE 33
Choose who to work with. It’s tempting to take a position because of the projects you’ll ...

> Merging 1 nodes into parent node.
> Parent node id: a36b069b-6fb2-4058-a393-ca6e2ee181da.
> Parent node text: PAGE 23
Each project is only one step on a longer journey, hopefully one that has a positive impa...

> Merging 1 nodes into parent node.
> Parent node id: 4560154b-606a-46c1-9521-15f101585ea1.
> Parent node text: PAGE 29
If you’re preparing to s

RuntimeError: generator raised StopIteration

In [41]:
from trulens.core import Tru 

Tru().get_leaderboard(app_ids=[])

,,latency,total_cost
app_name,app_version,,


In [42]:
Tru().run_dashboard()

Starting dashboard ...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at http://localhost:62668 .


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>

### Three layers

In [16]:
auto_merging_index_1 = build_automerging_index(
    documents,
    llm=llm,
    embed_model="local:BAAI/bge-small-en-v1.5",
    save_dir="merging_index_1",
    chunk_sizes=[2048,512,128],
)

In [17]:
auto_merging_engine_1 = get_automerging_query_engine(
    auto_merging_index_1,
    similarity_top_k=12,
    rerank_top_n=6,
)


In [18]:
tru_recorder = get_prebuilt_trulens_recorder(
    auto_merging_engine_1,
    app_id ='app_1'
)

instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.embeddings.multi_modal_base.MultiModalEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.base.embeddings.base.BaseEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.schema.TransformComponent'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.schema.BaseComponent'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'pydantic.main.BaseModel'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base

In [19]:
run_evals(eval_questions, tru_recorder, auto_merging_engine_1)

> Merging 4 nodes into parent node.
> Parent node id: bfab6d49-93d7-49ef-9680-fd6eb7387d69.
> Parent node text: PAGE 26
If you’re considering a role switch, a startup can be an easier place to do it than a big...

> Merging 4 nodes into parent node.
> Parent node id: 11893045-bbdd-4c17-8f7e-ba45178a0933.
> Parent node text: PAGE 25
Finding a job has a few predictable steps that include selecting the companies to which y...

> Merging 1 nodes into parent node.
> Parent node id: 27a30e9d-48b7-4bb3-8eaf-f33d3d730f52.
> Parent node text: PAGE 26
If you’re considering a role switch, a startup can be an easier place to do it than a big...

> Merging 1 nodes into parent node.
> Parent node id: 0b27ab0b-83b0-49e8-8860-dbc1d8098519.
> Parent node text: PAGE 25
Finding a job has a few predictable steps that include selecting the companies to which y...

> Merging 5 nodes into parent node.
> Parent node id: 491896be-d4f4-4cfd-a2ae-5bb537b4e111.
> Parent node text: PAGE 27
There’s a lot we don’t k

In [20]:
from trulens_eval import Tru

Tru().get_leaderboard(app_ids=[])

,,Answer Relevance,Context Relevance,Groundedness,latency,total_cost
app_name,app_version,,,,,
app_0,base,1.000000,0.944444,0.943297,21.122708,0.0
app_1,base,0.982456,0.824561,0.942786,18.281539,0.0


In [21]:
Tru().run_dashboard()

Starting dashboard ...
Dashboard already running at path:   Local URL: http://localhost:53253



<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>